# مختبر الأدوات واستدعاء الوظائف

في هذا المختبر، ستنفذ المفاهيم التي تعلمتها حول الأدوات واستدعاء الوظائف. ستقوم ب:
- تعريف أدوات بسيطة
- بناء سجل أدوات وموزع استدعاء
- اختبار السجل
- استخدام LangChain لتعريف أداة مع تحديد الخطأ وإصلاحه
- بناء حلقة وكيل تستخدم نموذج DeepSeek لاستدعاء الأدوات

جميع البيانات عشوائية باستخدام numpy و seed ثابت لضمان التكاثرية.

## 1. مفهوم الأدوات (Tools Concept)

الأدوات هي دوال تسمح للوكيل بالتفاعل مع العالم الخارجي، مثل جلب بيانات أو إجراء حسابات. عندما لا تكفي معرفة النموذج اللغوي الداخلية، يمكنه استدعاء أداة للحصول على معلومات محدثة.

In [ ]:
# استيراد المكتبات
import numpy as np

# تثبيت البذرة العشوائية للحصول على نتائج قابلة للتكرار
np.random.seed(42)

def add(a: float, b: float) -> float:
    # TODO: نفذ العملية وأعد الناتج
    pass

def subtract(a: float, b: float) -> float:
    # TODO: نفذ العملية وأعد الناتج
    pass

def multiply(a: float, b: float) -> float:
    # TODO: نفذ العملية وأعد الناتج
    pass

def get_weather(city: str) -> str:
    # TODO: أعد حالة طقس وهمية للمدينة المطلوبة باستخدام np.random لدرجة الحرارة
    pass

## 2. سجل الأدوات وموزع الاستدعاء (Tool Registry & Dispatcher)

سنقوم الآن بإنشاء قاموس يحوي أسماء الأدوات ومراجعها، ثم نكتب دالة `run_tool` التي تستقبل اسم الأداة ومعاملاتها وتقوم بتنفيذها.

In [ ]:
# سجل الأدوات
TOOLS = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "get_weather": get_weather
}

def run_tool(tool_name: str, **kwargs) -> str:
    """
    ينفذ الأداة المطلوبة مع المعاملات ويعيد النتيجة كسلسلة نصية.
    """
    # TODO: استخرج الأداة من السجل واستدعها، ثم أعد النتيجة كنص
    pass

In [ ]:
# اختبار run_tool مع بعض الحالات
print(run_tool("add", a=5, b=3))  # المفروض يطبع 8
print(run_tool("subtract", a=10, b=4))  # المفروض يطبع 6
print(run_tool("multiply", a=7, b=2))  # المفروض يطبع 14
print(run_tool("get_weather", city="Cairo"))  # يطبع حالة طقس عشوائية

## 3. استدعاء الدوال في LangChain (Function Calling)

الآن سنستخدم LangChain مع نموذج DeepSeek. نريد تعريف أداة الطقس باستخدام decorator `@tool`، لكن التعريف التالي يحتوي على خطأ يمنع عمله بشكل صحيح. اكتشف الخطأ وأصلحه.

In [ ]:
from langchain.tools import tool

@tool
def get_weather(city: str):
    """Get the current weather for a city."""
    # محاكاة بيانات الطقس
    import numpy as np
    np.random.seed(42)  # نثبت البذرة للثبات
    temp = np.random.randint(10, 40)
    return f"The weather in {city} is {temp}°C"

# TODO: اكتشف الخطأ في تعريف الأداة أعلاه وقم بتصحيحه

## 4. دمج نتائج الأدوات في حلقة الوكيل (Tool Integration)

الآن سنبني حلقة وكيل بسيطة: نعطي المستخدم رسالة، ثم نستدعي النموذج مع الأدوات. إذا اقترح النموذج استدعاء أداة، ننفذها ونعيد النتيجة إلى النموذج للحصول على الإجابة النهائية.

In [ ]:
from langchain.chat_models import ChatDeepSeek
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
import os

# التأكد من توفر المفتاح
if "DEEPSEEK_API_KEY" not in os.environ:
    raise ValueError("لم يتم العثور على DEEPSEEK_API_KEY في متغيرات البيئة")

# إعداد النموذج مع الأدوات
model = ChatDeepSeek(model="deepseek-chat", api_key=os.environ["DEEPSEEK_API_KEY"])
model_with_tools = model.bind_tools([get_weather])

# رسالة المستخدم
user_query = "ما هو الطقس في اسطنبول؟"
messages = [
    SystemMessage(content="أنت مساعد مفيد. استخدم المعلومات المتاحة."),
    HumanMessage(content=user_query)
]

response = model_with_tools.invoke(messages)

# TODO: افحص response، إذا كان يحتوي على استدعاء أداة (tool_calls)، قم بتنفيذ الأداة وإعادة استدعاء النموذج للحصول على الإجابة النهائية.

In [ ]:
# اختبار الوكيل مع الاستعلام (سيتم ملؤه بعد تنفيذ TODO)
print("إجابة الوكيل:", result)

## 5. الخلاصة

في هذا المختبر، تدربت على:
- تعريف أدوات بسيطة وسجل لاستدعائها
- كتابة موزع استدعاء يدوي
- اكتشاف الأخطاء في تعريف أداة LangChain
- بناء حلقة وكيل تستخدم نموذجًا لغويًا مع استدعاء الأدوات

يمكنك توسيع هذا المختبر بإضافة المزيد من الأدوات مثل الآلة الحاسبة (Calculator) أو البحث في قاعدة بيانات محلية.